In [85]:
import math
import numpy as np

### 1. Método de Chun

A continuación, se define el método de Chun de la siguiente manera:

$$
\begin{aligned}
y_k &= x_k - \frac{f(x_k)}{f'(x_k)} \\
x_{k+1} &= y_k - \frac{f(x_k) + 2f(y_k)}{f(x_k)} \frac{f(y_k)}{f'(x_k)}
\end{aligned}
$$

### a) Programar el método de Chun en Python.  

In [86]:
def Chun(f, df, x0, tol, maxiter=30):
    iters = 0
    error = tol + 1
    error_x = list()


    while error > tol and iters <= maxiter:
        y = x0 - (f(x0)/df(x0))
        x = y - (((f(x0)+2*f(y))/f(x0)) * (f(y)/df(x0)))
        error = math.fabs(x - x0)
        error_x.append(error)
        x0 = x
        iters = iters + 1

    error_x = np.array(error_x)

    if len(error_x) >= 3:
        ACOC = np.log(error_x[2:] / error_x[1:-1]) / np.log(error_x[1:-1] / error_x[0:-2])
    else:
        ACOC = None

    return [x,iters,error,ACOC]

b) Resolver la ecuación utilizando el método de Chun la ecuación $(x - 1)^3 - 1 + e^x = 0$, utilizando una estimación de $x_0 = 0.1$ y la tolerancia de $10^{-7}$. Obtener la solución aproximada, el error, la cantidad de iteraciones realizadas y el ACOC.

In [87]:
f = lambda x: ((x-1)**3)-1+math.exp(x)
df = lambda x: (3*(x-1)**2) + math.exp(x)
x0 = 0.1
tol = 1e-10

x, iters, error, ACOC = Chun(f, df, x0, tol)

print(f"Solución aproximada: {x}")
print(f"Iteraciones realizadas: {iters}")
print(f"Error final: {error}")
print(f"ACOC: {ACOC}")

Solución aproximada: 0.2975706656987736
Iteraciones realizadas: 3
Error final: 1.3433698597964394e-13
ACOC: [4.02460381]


### 2. Método SH4

Se considera el Método SH4 definido como:

$$
\begin{aligned}
y_k &= x_k - \frac{f(x_k)}{f'(x_k)} \\
z_k &= y_k - \frac{f(x_k)}{f(x_k) - 2f(y_k)} \frac{f(y_k)}{f'(x_k)} \\
x_{k+1} &= z_k - \frac{f(x_k) + 2f(y_k)}{f(x_k) + 4f(y_k)} \frac{f(z_k)}{f'(x_k)}
\end{aligned}
$$

a) Programar el método de SH4 en Python.

In [88]:
def SH4(f,df,x0,tol,maxiter=30):
    iters = 0
    error = tol + 1
    error_list = list()

    while error > tol and iters <= maxiter:
        y = x0 - (f(x0)/df(x0))
        z = y - f(x0)/(f(x0)-2*f(y)) * (f(y)/df(x0))
        x = z - (f(x0)+2*f(y))/(f(x0)+4*f(y)) * (f(z)/df(x0))

        error = math.fabs(x-x0)
        error_list.append(error)
        x0 = x
        iters += 1
        
    error_list = np.array(error_list)
    ACOC = np.log(error_list[2:]/error_list[1:-1])/np.log(error_list[1:-1]/error_list[0:-2])
    
    return [x,iters,error,ACOC]

b) Resolver la ecuación utilizando el método SH4 la ecuación $x^2 - x e^x + \cos(x) = 0$ utilizando una estimación de $x_0 = 1$ y la tolerancia de $10^{-8}$. Obtener la solución aproximada, el error, la cantidad de iteraciones realizadas y el ACOC.

In [89]:
f = lambda x: x**2 - x*math.exp(x) + math.cos(x)
df = lambda x: 2*x - math.exp(x) - x*math.exp(x) - math.sin(x)
x0 = 1
tol = 1e-8

x,iters,error,ACOC = SH4(f,df,x0,tol)

print(f"Solución aproximada: {x}")
print(f"Iteraciones realizadas: {iters}")
print(f"Error final: {error}")
print(f"ACOC: {ACOC}")

Solución aproximada: 0.6391540963320076
Iteraciones realizadas: 3
Error final: 1.5543122344752192e-15
ACOC: [4.8687908]


### 5. Método de Trapecios para Sistemas No Lineales

Otro método iterativo utilizado para resolver sistemas no lineales es el **Método** de Trapecios:

$$
\begin{aligned}
\mathbf{y}^{(k)} &= \mathbf{x}^{(k)} - J^{-1}(\mathbf{x}^{(k)}) F(\mathbf{x}^{(k)}) \\
\mathbf{x}^{(k+1)} &= \mathbf{x}^{(k)} - 2 \left[ J(\mathbf{y}^{(k)}) + J(\mathbf{x}^{(k)}) \right]^{-1} F(\mathbf{x}^{(k)})
\end{aligned}
$$

a) Diseñar el Método de Trapecios en Python.  

In [90]:
def TrapeciosSis(F, JF, x0, tol, maxiter=30):
    """
    Implementación del Método de Trapecios para sistemas no lineales.
    """
    iters = 0
    error = tol + 1
    while error > tol and iters <= maxiter:
        # Paso 1: Resolver J(x0) * z = F(x0)
        z = np.linalg.solve(JF(x0), F(x0))
        y = x0 - z
        
        # Paso 2: Resolver [J(y) + J(x0)] * w = F(x0)
        w = np.linalg.solve(JF(y) + JF(x0), F(x0))
        x = x0 - 2 * w
        
        # Criterio de paro mediante norma espectral/euclidiana
        error = np.linalg.norm(x - x0)
        x0 = x
        iters = iters + 1
        
    return [x, iters, error]

b) Con el Método de Trapecios resolver el sistema no lineal:

$$
\begin{aligned}
2x + 3y - 4 &= 0 \\
e^{-x} - y^2 + \cos(x) &= 0
\end{aligned}
$$

Considerando la estimación inicial $\mathbf{x}^{(0)} = (-1, 1)^T$ con una tolerancia de $10^{-10}$.

In [91]:
# Definición del sistema no lineal F(w) y su Jacobiana JF(w)
def F(w):
    x, y = w
    S = np.zeros(2)
    S[0] = 2*x + 3*y - 4
    S[1] = math.exp(-x) - y**2 + math.cos(x)
    return S

def JF(w):
    x, y = w
    M = np.zeros((2, 2))
    M[0, 0] = 2
    M[0, 1] = 3
    M[1, 0] = -math.exp(-x) - math.sin(x)
    M[1, 1] = -2*y
    return M

# Parámetros del problema
x0 = np.array([-1.0, 1.0])
tol = 1e-10
maxiter = 30

# Ejecución
solucion, iters, error_final = TrapeciosSis(F, JF, x0, tol, maxiter)

print(f"Solución aproximada: {solucion}")
print(f"Iteraciones realizadas: {iters}")
print(f"Error final: {error_final}")

Solución aproximada: [0.66683988 0.88877341]
Iteraciones realizadas: 31
Error final: 0.000171362324292469


### 6. Método de Simpson para Sistemas No Lineales

El método de Simpson para resolver sistemas no lineales está definido como:

$$
\begin{aligned}
\mathbf{y}^{(k)} &= \mathbf{x}^{(k)} - J^{-1}(\mathbf{x}^{(k)}) F(\mathbf{x}^{(k)}) \\
\mathbf{x}^{(k+1)} &= \mathbf{x}^{(k)} - 6 \left[ J(\mathbf{x}^{(k)}) + 4 J\left( \frac{\mathbf{x}^{(k)} + \mathbf{y}^{(k)}}{2} \right) + J(\mathbf{y}^{(k)}) \right]^{-1} F(\mathbf{x}^{(k)})
\end{aligned}
$$

a) Implementar el Método de Simpson en Python. 

In [92]:
import numpy as np
import math

def Simpson(F, JF, x0, tol, maxiter=30):
    iters = 0
    error = tol + 1
    x = x0

    while error > tol and iters < maxiter:
        z = np.linalg.solve(JF(x0), F(x0))
        y = x0 - z

        i = JF(x0) + 4*JF((x0 + y)/2) + JF(y)
        w = np.linalg.solve(i, F(x0))
        x = x0 - 6*w

        error = np.linalg.norm(x - x0)
        x0 = x
        iters += 1

    return [x, iters, error]


b) Resolver el sistema no lineal con el método de Simpson:

$$
\begin{aligned}
4x^2 + y^2 - 4 &= 0 \\
x + y - \sin(x - y) &= 0
\end{aligned}
$$

Usando una estimación inicial $\mathbf{x}^{(0)} = (0, 0)^T$ y establecer la tolerancia de $10^{-9}$.

In [93]:
def F(w):
    x, y = w
    S = np.zeros(2)
    S[0] = 4*x**2 + y**2 - 4
    S[1] = x + y - math.sin(x - y)
    return S

def JF(w):
    x, y = w
    M = np.zeros((2, 2))
    M[0, 0] = 8*x
    M[0, 1] = 2*y
    M[1, 0] = 1 - math.cos(x - y)
    M[1, 1] = 1 + math.cos(x - y)
    return M

tol = 1e-9
x0 = np.array([1.0, -0.1])

x, iters, error = Simpson(F, JF, x0, tol)

print(f"Solución aproximada: {x}")
print(f"Iteraciones realizadas: {iters}")
print(f"Error final: {error}")
# print(f"||F(x)|| = {np.linalg.norm(F(x))}")


Solución aproximada: [ 0.99860694 -0.10553049]
Iteraciones realizadas: 3
Error final: 1.5700924586837752e-16


### 7. Sistema No Lineal y Métodos Comparativos

Se considera el sistema no lineal:

$$
\begin{aligned}
3x^2 - y^2 &= 0 \\
3xy^2 - x^3 - 1 &= 0
\end{aligned}
$$

a) Obtener la matriz jacobiana asociada al sistema no lineal de ecuaciones.  

b) Resolver el sistema anterior usando los métodos de Newton, Simpson y Trapecios usando la estimación inicial $\mathbf{x}^{(0)} = (1, 1)^T$ y una tolerancia de $10^{-6}$. Obtener por cada método la solución aproximada, el error, la cantidad de iteraciones realizadas.
